# Classic RBM vs BEBM-RBMEnergy: log-likelihood with AIS

This notebook compares the last checkpoint of the classic RBM and the last checkpoint of the BEBM trained with an RBM visible energy. Both models are evaluated as `BBRBM` objects so that AIS uses the fast RBM implementation in `rbms.partition_function.ais`.


In [ ]:
import torch

from rbms.EBM_binary import BEBM
from rbms.bernoulli_bernoulli import BBRBM
from rbms.dataset import load_dataset
from rbms.io import load_params
from rbms.partition_function.ais import compute_partition_function_ais
from rbms.utils import get_saved_updates

device = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype = torch.float32

train_dataset_name = "data/MNIST_train.h5"
test_dataset_name = "data/MNIST_test.h5"

MODEL_FILES = {
    "Classic RBM | PCD-10": "pcd_trains/BBRBM_MNIST_PCD10_h500_ch1024_lr1e-2_100k.h5",
    "BEBM RBMEnergy | DMALA-100": "pcd_trains/BEBM_RBMEnergy_MNIST_DMALA100_h500_ch1024_lr1e-2_100k.h5",
}

AIS_NUM_CHAINS = 1000
AIS_NUM_BETA = 2000
LL_BATCH_SIZE = 2048

print("device:", device)
for label, filename in MODEL_FILES.items():
    print(label, "->", filename)


In [ ]:
@torch.no_grad()
def as_bbrbm(params):
    if isinstance(params, BBRBM):
        return params

    if isinstance(params, BEBM):
        energy = params.energy
        required = ("weight", "vbias", "hbias")
        if all(hasattr(energy, name) for name in required):
            return BBRBM(
                weight_matrix=energy.weight.detach().clone(),
                vbias=energy.vbias.detach().clone(),
                hbias=energy.hbias.detach().clone(),
                device=params.device,
                dtype=params.dtype,
            )

    raise TypeError("Could not convert parameters to BBRBM.")


@torch.no_grad()
def mean_log_likelihood(params, data, log_z, batch_size=2048):
    total_log_prob = 0.0
    num_samples = data.shape[0]

    for start in range(0, num_samples, batch_size):
        batch = data[start : start + batch_size].to(device=params.device, dtype=params.dtype)
        energy = params.compute_energy_visibles(batch)
        total_log_prob += (-energy - log_z).sum().item()

    return total_log_prob / num_samples


train_dataset, test_dataset = load_dataset(
    dataset_name=train_dataset_name,
    test_dataset_name=test_dataset_name,
    device=device,
    dtype=dtype,
)

train_visible = train_dataset.data.to(device=device, dtype=dtype)
test_visible = test_dataset.data.to(device=device, dtype=dtype)

models = {}
for label, filename in MODEL_FILES.items():
    last_update = int(get_saved_updates(filename)[-1])
    params = load_params(filename, last_update, device=device, dtype=dtype)
    models[label] = {
        "filename": filename,
        "last_update": last_update,
        "params_original": params,
        "params_rbm": as_bbrbm(params),
    }

    print(label)
    print("  last update:", last_update)
    print("  original type:", type(params).__name__)
    print("  AIS type:     ", type(models[label]["params_rbm"]).__name__)


In [ ]:
ais_log_z = {}

for label, model in models.items():
    print(f"Computing AIS log Z for {label}...")
    log_z = compute_partition_function_ais(
        num_chains=AIS_NUM_CHAINS,
        num_beta=AIS_NUM_BETA,
        params=model["params_rbm"],
    )
    ais_log_z[label] = log_z
    print(f"  AIS log Z = {log_z:.6f}")


In [ ]:
ll_results = {}

for label, model in models.items():
    params_rbm = model["params_rbm"]
    log_z = ais_log_z[label]

    train_ll = mean_log_likelihood(
        params=params_rbm,
        data=train_visible,
        log_z=log_z,
        batch_size=LL_BATCH_SIZE,
    )
    test_ll = mean_log_likelihood(
        params=params_rbm,
        data=test_visible,
        log_z=log_z,
        batch_size=LL_BATCH_SIZE,
    )

    ll_results[label] = {
        "last_update": model["last_update"],
        "log_z": log_z,
        "train_mean_ll": train_ll,
        "test_mean_ll": test_ll,
        "train_nll": -train_ll,
        "test_nll": -test_ll,
    }

for label, result in ll_results.items():
    print(label)
    print(f"  update:         {result['last_update']}")
    print(f"  AIS log Z:      {result['log_z']:.6f}")
    print(f"  train mean LL:  {result['train_mean_ll']:.6f} nats")
    print(f"  test mean LL:   {result['test_mean_ll']:.6f} nats")
    print(f"  train NLL:      {result['train_nll']:.6f} nats")
    print(f"  test NLL:       {result['test_nll']:.6f} nats")
    print()


In [ ]:
for key in ["log_z", "train_mean_ll", "test_mean_ll", "train_nll", "test_nll"]:
    print(key)
    for label, result in ll_results.items():
        print(f"  {label}: {result[key]:.6f}")
    print()
